# Tutorial 12: Differential Evolution

**Prerequisites:** T11 (Particle Swarm Optimization)  
**Up Next:** T13 — Analytical Gradients

---

## Concept

**Differential Evolution (DE)** is an elegantly simple global optimizer. Like GA, it maintains a population, but the mutation mechanism is unique: new candidates are created by adding *scaled differences* between existing population members. This automatically adapts the step size to the current spread of the population.

The classic DE/rand/1/bin variant uses three steps:
1. **Mutation:** pick three distinct individuals $\mathbf{a}, \mathbf{b}, \mathbf{c}$; form donor $\mathbf{v} = \mathbf{a} + F(\mathbf{b} - \mathbf{c})$.
2. **Crossover:** mix donor with the target individual gene-by-gene.
3. **Selection:** keep whichever is better — the original or the trial vector.

## Key Takeaway

> **DE is simple, robust, and often the go-to global method. Its self-adapting step size (from population differences) and greedy selection make it effective with very few tuning parameters.**

## Math Core

For each target vector $\mathbf{x}_i$ in the population:

**Mutation (rand/1):** select $a, b, c$ randomly ($\neq i$),
$$\mathbf{v}_i = \mathbf{x}_a + F\,(\mathbf{x}_b - \mathbf{x}_c)$$

**Binomial crossover:** for each dimension $j$,
$$u_{i,j} = \begin{cases} v_{i,j} & \text{if } r_j < CR \text{ or } j = j_{\text{rand}} \\ x_{i,j} & \text{otherwise} \end{cases}$$

**Selection:**
$$\mathbf{x}_i^{(t+1)} = \begin{cases} \mathbf{u}_i & \text{if } f(\mathbf{u}_i) \le f(\mathbf{x}_i) \\ \mathbf{x}_i & \text{otherwise} \end{cases}$$

Parameters: $F \in [0.4, 1.0]$ (scale factor), $CR \in [0, 1]$ (crossover rate).

## Code

In [ ]:
from dms.mechanisms.fourbar import FourBar
from dms.curves import CompareCurves
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

L0, L1 = 1.0, 1.0
PX, PY = 0.5, 0.3
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### Objective landscape

In [ ]:
l2_vals = np.linspace(0.3, 2.5, 40)
l3_vals = np.linspace(0.3, 2.5, 40)
L2g, L3g = np.meshgrid(l2_vals, l3_vals)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2g, L3g)

### Implementing DE from scratch

In [ ]:
def run_de(objective, bounds, pop_size=40, n_gen=80,
           F=0.8, CR=0.9, seed=42):
    """Differential Evolution (DE/rand/1/bin)."""
    rng = np.random.default_rng(seed)
    n_dim = len(bounds)
    lo = np.array([b[0] for b in bounds])
    hi = np.array([b[1] for b in bounds])

    # Initialize population
    pop = rng.uniform(lo, hi, size=(pop_size, n_dim))
    fit = np.array([objective(ind) for ind in pop])

    best_history = [fit.min()]
    pop_history = [pop.copy()]

    for gen in range(n_gen):
        for i in range(pop_size):
            # Pick three distinct individuals != i
            idxs = [j for j in range(pop_size) if j != i]
            a, b, c = rng.choice(idxs, size=3, replace=False)

            # Mutation: donor vector
            donor = pop[a] + F * (pop[b] - pop[c])
            donor = np.clip(donor, lo, hi)

            # Binomial crossover
            trial = pop[i].copy()
            j_rand = rng.integers(0, n_dim)
            for j in range(n_dim):
                if rng.random() < CR or j == j_rand:
                    trial[j] = donor[j]

            # Selection
            f_trial = objective(trial)
            if f_trial <= fit[i]:
                pop[i] = trial
                fit[i] = f_trial

        best_history.append(fit.min())
        pop_history.append(pop.copy())

    best_idx = np.argmin(fit)
    return pop[best_idx], fit[best_idx], best_history, pop_history

In [ ]:
bounds = [(0.3, 2.5), (0.3, 2.5)]
best_x, best_f, history_de, pop_history = run_de(objective, bounds)

print(f'DE best solution: l2={best_x[0]:.4f}, l3={best_x[1]:.4f}')
print(f'DE best objective: {best_f:.4f}')

### Convergence plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_de, 'b-', lw=1.5)
ax.set_xlabel('Generation')
ax.set_ylabel('Best objective')
ax.set_title('DE convergence')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()

### Population on contour over generations

In [ ]:
snap_gens = [0, 20, 50, len(pop_history) - 1]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, g in zip(axes, snap_gens):
    ax.contourf(L2g, L3g, np.log1p(Z), levels=30, cmap='viridis', alpha=0.8)
    pop_g = pop_history[g]
    ax.scatter(pop_g[:, 0], pop_g[:, 1], c='red', s=15, edgecolors='white', linewidths=0.5)
    ax.set_title(f'Gen {g}')
    ax.set_xlabel(r'$\ell_2$')
    ax.set_ylabel(r'$\ell_3$')
    ax.set_xlim(0.3, 2.5)
    ax.set_ylim(0.3, 2.5)

fig.suptitle('DE population snapshots', fontsize=14)
plt.tight_layout()

### Comparing DE vs GA vs PSO

Let's run all three methods with the same budget (40 individuals, 80 generations) and compare their convergence.

In [ ]:
def run_ga(objective, bounds, pop_size=40, n_gen=80,
           tournament_k=3, crossover_alpha=0.5,
           mutation_rate=0.3, mutation_sigma=0.15, seed=42):
    """Simple real-coded GA (from Tutorial 10)."""
    rng = np.random.default_rng(seed)
    n_dim = len(bounds)
    lo = np.array([b[0] for b in bounds])
    hi = np.array([b[1] for b in bounds])
    pop = rng.uniform(lo, hi, size=(pop_size, n_dim))
    fitness = np.array([objective(ind) for ind in pop])
    best_history = [fitness.min()]
    for gen in range(n_gen):
        new_pop = []
        best_idx = np.argmin(fitness)
        new_pop.append(pop[best_idx].copy())
        while len(new_pop) < pop_size:
            parents = []
            for _ in range(2):
                candidates = rng.integers(0, pop_size, size=tournament_k)
                winner = candidates[np.argmin(fitness[candidates])]
                parents.append(pop[winner])
            p1, p2 = parents
            beta = rng.uniform(-crossover_alpha, 1 + crossover_alpha, size=n_dim)
            child = p1 + beta * (p2 - p1)
            for j in range(n_dim):
                if rng.random() < mutation_rate:
                    child[j] += rng.normal(0, mutation_sigma)
            child = np.clip(child, lo, hi)
            new_pop.append(child)
        pop = np.array(new_pop[:pop_size])
        fitness = np.array([objective(ind) for ind in pop])
        best_history.append(fitness.min())
    best_idx = np.argmin(fitness)
    return pop[best_idx], fitness[best_idx], best_history


def run_pso(objective, bounds, n_particles=40, n_iter=80,
            w=0.7, c1=1.5, c2=1.5, seed=42):
    """PSO (from Tutorial 11)."""
    rng = np.random.default_rng(seed)
    n_dim = len(bounds)
    lo = np.array([b[0] for b in bounds])
    hi = np.array([b[1] for b in bounds])
    span = hi - lo
    pos = rng.uniform(lo, hi, size=(n_particles, n_dim))
    vel = rng.uniform(-0.2 * span, 0.2 * span, size=(n_particles, n_dim))
    fit = np.array([objective(p) for p in pos])
    pbest_pos = pos.copy()
    pbest_fit = fit.copy()
    gbest_idx = np.argmin(fit)
    gbest_pos = pos[gbest_idx].copy()
    gbest_fit = fit[gbest_idx]
    best_history = [gbest_fit]
    for it in range(n_iter):
        r1 = rng.random(size=(n_particles, n_dim))
        r2 = rng.random(size=(n_particles, n_dim))
        vel = w * vel + c1 * r1 * (pbest_pos - pos) + c2 * r2 * (gbest_pos - pos)
        pos = pos + vel
        pos = np.clip(pos, lo, hi)
        fit = np.array([objective(p) for p in pos])
        improved = fit < pbest_fit
        pbest_pos[improved] = pos[improved]
        pbest_fit[improved] = fit[improved]
        gen_best_idx = np.argmin(pbest_fit)
        if pbest_fit[gen_best_idx] < gbest_fit:
            gbest_pos = pbest_pos[gen_best_idx].copy()
            gbest_fit = pbest_fit[gen_best_idx]
        best_history.append(gbest_fit)
    return gbest_pos, gbest_fit, best_history

In [ ]:
ga_x, ga_f, ga_hist = run_ga(objective, bounds)
pso_x, pso_f, pso_hist = run_pso(objective, bounds)

print(f'GA  best: l2={ga_x[0]:.4f}, l3={ga_x[1]:.4f}, f={ga_f:.4f}')
print(f'PSO best: l2={pso_x[0]:.4f}, l3={pso_x[1]:.4f}, f={pso_f:.4f}')
print(f'DE  best: l2={best_x[0]:.4f}, l3={best_x[1]:.4f}, f={best_f:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ga_hist, 'r-', lw=1.5, label=f'GA (best={ga_f:.4f})')
ax.plot(pso_hist, 'g-', lw=1.5, label=f'PSO (best={pso_f:.4f})')
ax.plot(history_de, 'b-', lw=1.5, label=f'DE (best={best_f:.4f})')
ax.set_xlabel('Generation / Iteration')
ax.set_ylabel('Best objective')
ax.set_title('DE vs GA vs PSO — convergence comparison')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

### scipy.optimize.differential_evolution — the production version

For real work, use SciPy's well-tuned implementation.

In [ ]:
from scipy.optimize import differential_evolution

result = differential_evolution(objective, bounds, seed=42,
                                 maxiter=80, popsize=15, tol=0)

print(f'scipy DE solution: l2={result.x[0]:.4f}, l3={result.x[1]:.4f}')
print(f'scipy DE objective: {result.fun:.4f}')
print(f'Function evaluations: {result.nfev}')

### Visualizing the DE result

In [ ]:
path_de = np.array([coupler_point(t, best_x[0], best_x[1]) for t in THETAS])

mechanism = FourBar(L0, L1, best_x[0], best_x[1])
fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
ax.plot(path_de[:, 0], path_de[:, 1], 'b.-', label='coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'DE result: $\\ell_2$={best_x[0]:.3f}, $\\ell_3$={best_x[1]:.3f}, f={best_f:.4f}')
plt.tight_layout()

---

## Exercises

1. **Scale factor F:** Run DE with $F = 0.3$ and $F = 1.5$. How does $F$ affect exploration vs. exploitation?

2. **Crossover rate CR:** Try $CR = 0.1$ (little crossover) vs $CR = 1.0$ (full crossover). Which works better for this 2D problem?

3. **DE strategies:** Modify the mutation to use DE/best/1 (replace $\mathbf{x}_a$ with the population's best individual). Does this speed up convergence? Does it hurt diversity?

4. **Hybrid DE + BFGS:** Take the best DE solution and polish it with BFGS. Compare the result to using `scipy.optimize.differential_evolution` with `polish=True`.